# Train the Arabizi → Arabic model on Colab
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
Checkpoints are saved to your Google Drive every epoch, so a disconnected session loses at most one epoch: run the notebook again and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/arabizi'
!mkdir -p $DRIVE/runs

In [ ]:
# the code and data, from GitHub (or upload the folder to Drive and cd there instead)
!git clone -q https://github.com/BackendNeuron/arabizi.git /content/arabizi || (cd /content/arabizi && git pull -q)
%cd /content/arabizi
!pip -q install -r requirements.txt
# keep runs/ on Drive so checkpoints survive the session
!rm -rf runs && ln -s $DRIVE/runs runs

In [ ]:
!python tools/build_dataset.py --variants 4

In [ ]:
import os
# resume if a previous session was cut off, otherwise start fresh
mode = '--resume' if os.path.exists('runs/last.pt') else '--new'
!python train.py $mode --epochs 40

In [ ]:
!python tools/plots.py --model runs/best.pt
from IPython.display import Image, display
for n in ['accuracy', 'cer', 'loss', 'topics']:
    display(Image(f'docs/charts/{n}-light.png'))

## Later: continue training on new data
Add sentences to `data/seed/`, rebuild the dataset, then continue from the best model at a lower learning rate:
```
!python tools/build_dataset.py
!python train.py --continue-from runs/best.pt --epochs 20
```